# Lesson 6: Teams of agents

For each cell: **guess what it will print first**, then press ▶ and check.

From [Build an AI Agent](https://gradlabs.in/learn/agents/lessons/lesson6.html).

In [ ]:
import re

In [ ]:
# The loop from Lesson 3, but each agent brings its own instructions and tools
def run_agent(instructions, question, tools, llm, step_limit=6):
    text = instructions + "\n\nQuestion: " + question + "\n"
    for step in range(step_limit):
        reply = llm(text)
        text += reply + "\n"
        if "ANSWER:" in reply:
            return reply.split("ANSWER:")[1].strip()
        act = re.search(r"ACT: (\w+)\((.*)\)", reply)
        if act is None or act.group(1) not in tools:
            look = "Error: write ACT: tool(input) with a tool you have"
        else:
            look = tools[act.group(1)](act.group(2))
        text += "LOOK: " + look + "\n"
    return "Stopped: the step limit was reached."

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# Make an agent: a name, its instructions, its tools, and its (fake) LLM's replies.
# make_agent makes a new function for each agent and hands it back.
# That function remembers its own name, instructions, tools and replies.
def make_agent(name, instructions, tools, replies):
    def fake_llm(text):        # this agent's fake LLM: hands back its next reply
        return replies.pop(0)
    def agent(task):
        print(f"  {name} gets: {task}")
        answer = run_agent(instructions, task, tools, fake_llm)
        print(f"  {name} answers: {answer}")
        return answer
    return agent

def search(words):
    return "Cherai Beach: calm water, 25 km from Kochi. Fort Kochi Beach: rocky."

researcher = make_agent("researcher", "You find facts. Tool: search(words).", {"search": search}, [
    "ACT: search(beaches near Kochi)",
    "ANSWER: Cherai Beach is best: calm water, 25 km from Kochi.",
])
writer = make_agent("writer", "You write short, friendly text. No tools.", {}, [
    "ANSWER: Cherai Beach is the best beach near Kochi: calm, and only 40 km away.",
    "ANSWER: Cherai Beach is the best beach near Kochi: calm, and only 25 km away.",
])
checker = make_agent("checker", "You check a draft against the facts. Answer OK, or FIX: and what's wrong.", {}, [
    "ANSWER: FIX: the facts say 25 km, not 40 km.",
    "ANSWER: OK",
])

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# The manager: its tools are the other agents
manager_replies = [
    "ACT: ask_researcher(Find the best beach near Kochi)",
    "ACT: ask_writer(Write one sentence: Cherai Beach, calm water, 25 km from Kochi)",
    "ACT: ask_checker(Facts: 25 km from Kochi. Draft: calm, and only 40 km away.)",
    "ACT: ask_writer(Fix it: the beach is 25 km away, not 40 km)",
    "ACT: ask_checker(Facts: 25 km from Kochi. Draft: calm, and only 25 km away.)",
    "ANSWER: Cherai Beach is the best beach near Kochi: calm, and only 25 km away.",
]
manager = make_agent("manager", "You lead a team. Tools: ask_researcher, ask_writer, ask_checker.",
                     {"ask_researcher": researcher, "ask_writer": writer, "ask_checker": checker},
                     manager_replies)

print(manager("Write one sentence about the best beach near Kochi."))

## Now a real LLM

The same code, with a small free model (Qwen2.5 1.5B) instead of the fake LLM.
The first run downloads it (about 3 GB), so give it a few minutes.

Small models make mistakes. Watch the `LOOK:` lines: errors go back to the LLM, so it can often fix them.

In [ ]:
# A real LLM: a small, free model that runs inside Colab
# (faster with a GPU: Runtime → Change runtime type → T4 GPU)
import torch                        # the library that does the model's math
from transformers import pipeline   # a ready-made way to load a model and use it

In [ ]:
# Load Qwen2.5 1.5B (about 3 GB, downloaded the first time).
# device=0 means "use the GPU", -1 means "use the CPU": it picks the GPU if there is one.
chat = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
                device=0 if torch.cuda.is_available() else -1)

def real_llm(text):
    # The chat gets a list with one message: a dictionary saying who it's from ("role")
    # and what it says ("content"). Here: all the text so far, from the user.
    # max_new_tokens=120: write at most 120 tokens (pieces of words).
    # do_sample=False: always pick the most likely next piece, so every run is the same.
    out = chat([{"role": "user", "content": text}], max_new_tokens=120, do_sample=False)
    # out[0]["generated_text"] is the whole chat, now with the model's reply at the end.
    # [-1] counts from the end: the last message. ["content"] is its text.
    reply = out[0]["generated_text"][-1]["content"]
    # Keep the reply only up to its first ACT: or ANSWER: line
    kept = []
    for line in reply.split("\n"):
        kept.append(line)
        if line.startswith("ACT:") or line.startswith("ANSWER:"):
            break   # cut here: the program writes the LOOK: line, not the LLM
    return "\n".join(kept)   # glue the kept lines back together, one per line

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# The same team, now with a real LLM for every agent
RULES = "\nTo use a tool, write one line: ACT: tool(input)\nWhen you're done, write: ANSWER: your answer"

def make_real_agent(name, instructions, tools):
    def agent(task):
        print(f"  {name} gets: {task}")
        answer = run_agent(instructions + RULES, task, tools, real_llm)
        print(f"  {name} answers: {answer}")
        return answer
    return agent

researcher = make_real_agent("researcher", "You find facts. Tool: search(words).", {"search": search})
writer = make_real_agent("writer", "You write short, friendly text. No tools.", {})
checker = make_real_agent("checker", "You check a draft against the facts. Answer OK, or FIX: and what's wrong.", {})
manager = make_real_agent("manager", "You lead a team. Tools: ask_researcher(task), ask_writer(task), ask_checker(task).",
                          {"ask_researcher": researcher, "ask_writer": writer, "ask_checker": checker})
print(manager("Write one sentence about the best beach near Kochi."))